# Uji Statistik Tujuan 1 - Seleksi SLM

**Step 1**: Uji Konkordansi Spearman — validasi konsistensi IndoMMLU vs IndoCareer  
**Step 2**: One-Way ANOVA — uji signifikansi perbedaan 5 model teratas (didahului uji asumsi)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, shapiro, levene, f_oneway, kruskal, mannwhitneyu
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
from itertools import combinations

warnings.filterwarnings('ignore')

RESULT_PATH = "/kaggle/input/notebooks/muhammadnabilul/result-tujuan1/ranking_final_komprehensif.csv"

df = pd.read_csv(RESULT_PATH)

MMLU_COLS   = [c for c in df.columns if c.startswith("mmlu_")]
CAREER_COLS = [c for c in df.columns if c.startswith("career_")]
ALL_COLS    = MMLU_COLS + CAREER_COLS  

ALL_MODELS = df.sort_values("final_score_50_50", ascending=False)["model"].tolist()

print(f"List Model: {ALL_MODELS}")
print(f"Jumlah observasi per model: {len(ALL_COLS)} ({len(MMLU_COLS)} MMLU + {len(CAREER_COLS)} Career)")

List Model: ['Llama-3.1-8B', 'Llama3.2-3B', 'Gemma-2-2B', 'Mistral-7B', 'Qwen2.5-7B', 'Phi-3.5-mini', 'Llama3.2-1B', 'Qwen2.5-3B', 'Qwen2.5-1.5B', 'Qwen2.5-0.5B', 'SmolLM2-360M', 'SmolLM2-135M']
Jumlah observasi per model: 31 (25 MMLU + 6 Career)


---
## Step 1 — Uji Konkordansi Spearman

**Hipotesis**:  
- H₀: Tidak ada korelasi rank antara IndoMMLU dan IndoCareer (ρ = 0)  
- H₁: Terdapat korelasi rank positif yang signifikan (ρ > 0)  

**Tujuan**: Memvalidasi bahwa penggabungan IndoMMLU dan IndoCareer dengan bobot 50/50 dapat dipertanggungjawabkan secara statistik — kedua benchmark harus memberikan penilaian yang konsisten terhadap 12 model kandidat.

In [2]:
rho, p_spearman = spearmanr(df["indommlu_acc"], df["indocareer_acc"])

print("=" * 50)
print("UJI KONKORDANSI SPEARMAN")
print("=" * 50)
print(f"Spearman ρ  = {rho:.4f}")
print(f"p-value     = {p_spearman:.6f}")
print()
if p_spearman < 0.05:
    print(f"Kesimpulan: Terdapat korelasi rank yang signifikan (ρ={rho:.4f}, p={p_spearman:.4f} < 0.05).")
    print("IndoMMLU dan IndoCareer SEPAKAT dalam menentukan model terbaik.")
    print("→ Bobot 50/50 terjustifikasi secara statistik.")
else:
    print(f"Kesimpulan: Tidak terdapat korelasi rank yang signifikan (ρ={rho:.4f}, p={p_spearman:.4f} ≥ 0.05).")
    print("IndoMMLU dan IndoCareer TIDAK sepakat dalam menentukan model terbaik.")
    print("→ Bobot 50/50 perlu dikaji ulang.")

UJI KONKORDANSI SPEARMAN
Spearman ρ  = 0.9930
p-value     = 0.000000

Kesimpulan: Terdapat korelasi rank yang signifikan (ρ=0.9930, p=0.0000 < 0.05).
IndoMMLU dan IndoCareer SEPAKAT dalam menentukan model terbaik.
→ Bobot 50/50 terjustifikasi secara statistik.


---
## Step 2 — One-Way ANOVA (5 Model Teratas)

**Hipotesis**:  
- H₀: Rata-rata accuracy kelima model tidak berbeda secara signifikan  
- H₁: Terdapat setidaknya satu model yang berbeda secara signifikan  

**Data**: 31 nilai accuracy per model (25 subject IndoMMLU + 6 profession group IndoCareer)  
**Sebelum ANOVA**: harus dipenuhi asumsi normalitas (Shapiro-Wilk) dan homogenitas variansi (Levene)

In [3]:
scores = {}
for model in ALL_MODELS:
    row = df[df["model"] == model]
    vals = row[ALL_COLS].values.flatten().astype(float)
    scores[model] = vals

print(f"Jumlah observasi per model: {[len(v) for v in scores.values()]}")

Jumlah observasi per model: [31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31, 31]


### Step 2a — Uji Normalitas (Shapiro-Wilk)

- H₀: Data berdistribusi normal  
- H₁: Data tidak berdistribusi normal  
- Kriteria: p > 0.05 → asumsi normalitas terpenuhi

In [4]:
print("=" * 60)
print("UJI NORMALITAS SHAPIRO-WILK PER MODEL")
print("=" * 60)

normality_results = {}
all_normal = True
for model, vals in scores.items():
    stat, p = shapiro(vals)
    terpenuhi = p > 0.05
    normality_results[model] = terpenuhi
    if not terpenuhi:
        all_normal = False
    print(f"{model:<20} stat={stat:.4f}  p={p:.6f}  {'✓ Normal' if terpenuhi else '✗ Tidak Normal'}")

print()
print("Kesimpulan:", "Semua model memenuhi asumsi normalitas." if all_normal else "Ada model yang tidak memenuhi asumsi normalitas.")

UJI NORMALITAS SHAPIRO-WILK PER MODEL
Llama-3.1-8B         stat=0.9711  p=0.548918  ✓ Normal
Llama3.2-3B          stat=0.9603  p=0.297683  ✓ Normal
Gemma-2-2B           stat=0.9571  p=0.243901  ✓ Normal
Mistral-7B           stat=0.9752  p=0.671682  ✓ Normal
Qwen2.5-7B           stat=0.9738  p=0.628591  ✓ Normal
Phi-3.5-mini         stat=0.9757  p=0.684810  ✓ Normal
Llama3.2-1B          stat=0.9510  p=0.166403  ✓ Normal
Qwen2.5-3B           stat=0.9868  p=0.959876  ✓ Normal
Qwen2.5-1.5B         stat=0.9648  p=0.387980  ✓ Normal
Qwen2.5-0.5B         stat=0.9622  p=0.332287  ✓ Normal
SmolLM2-360M         stat=0.9640  p=0.371221  ✓ Normal
SmolLM2-135M         stat=0.9380  p=0.072640  ✓ Normal

Kesimpulan: Semua model memenuhi asumsi normalitas.


### Step 2b — Uji Homogenitas Variansi (Levene)

- H₀: Variansi kelima model homogen  
- H₁: Variansi kelima model tidak homogen  
- Kriteria: p > 0.05 → asumsi homogenitas terpenuhi

In [5]:
stat_lev, p_lev = levene(*scores.values())
homogen = p_lev > 0.05

print("=" * 50)
print("UJI HOMOGENITAS VARIANSI LEVENE")
print("=" * 50)
print(f"Levene stat = {stat_lev:.4f}")
print(f"p-value     = {p_lev:.6f}")
print()
print("Kesimpulan:", "Variansi homogen → asumsi terpenuhi." if homogen else "Variansi tidak homogen → asumsi tidak terpenuhi.")

UJI HOMOGENITAS VARIANSI LEVENE
Levene stat = 14.9709
p-value     = 0.000000

Kesimpulan: Variansi tidak homogen → asumsi tidak terpenuhi.


### Step 2c — Keputusan Uji

In [6]:
print("=" * 50)
print("KEPUTUSAN UJI")
print("=" * 50)
if all_normal and homogen:
    print("Semua asumsi terpenuhi → lanjut ONE-WAY ANOVA")
    uji = "anova"
else:
    print("Asumsi tidak terpenuhi → fallback KRUSKAL-WALLIS (non-parametrik)")
    if not all_normal:
        print(" - Normalitas tidak terpenuhi pada:", [m for m, v in normality_results.items() if not v])
    if not homogen:
        print(" - Homogenitas variansi tidak terpenuhi")
    uji = "kruskal"

KEPUTUSAN UJI
Asumsi tidak terpenuhi → fallback KRUSKAL-WALLIS (non-parametrik)
 - Homogenitas variansi tidak terpenuhi


### Step 2d — One-Way ANOVA atau Kruskal-Wallis

In [7]:
print("=" * 50)
if uji == "anova":
    print("ONE-WAY ANOVA")
    print("=" * 50)
    stat_f, p_anova = f_oneway(*scores.values())
    print(f"F-stat  = {stat_f:.4f}")
    print(f"p-value = {p_anova:.6f}")
    signifikan = p_anova < 0.05
    print()
    print("Kesimpulan:", "Terdapat perbedaan signifikan antar model (p < 0.05)." if signifikan else "Tidak terdapat perbedaan signifikan antar model (p ≥ 0.05).")
else:
    print("KRUSKAL-WALLIS")
    print("=" * 50)
    stat_kw, p_kruskal = kruskal(*scores.values())
    print(f"H-stat  = {stat_kw:.4f}")
    print(f"p-value = {p_kruskal:.6f}")
    signifikan = p_kruskal < 0.05
    print()
    print("Kesimpulan:", "Terdapat perbedaan signifikan antar model (p < 0.05)." if signifikan else "Tidak terdapat perbedaan signifikan antar model (p ≥ 0.05).")

KRUSKAL-WALLIS
H-stat  = 242.2137
p-value = 0.000000

Kesimpulan: Terdapat perbedaan signifikan antar model (p < 0.05).


### Step 2e — Post-hoc Test (jika signifikan)

In [8]:
if signifikan:
    print("=" * 60)
    if uji == "anova":
        print("POST-HOC: TUKEY HSD")
        print("=" * 60)
        all_vals   = np.concatenate(list(scores.values()))
        all_groups = np.concatenate([[m]*len(v) for m, v in scores.items()])
        tukey = pairwise_tukeyhsd(all_vals, all_groups, alpha=0.05)
        print(tukey)
    else:
        print("POST-HOC: MANN-WHITNEY (dengan Bonferroni correction)")
        print("=" * 60)
        import itertools
        pairs = list(itertools.combinations(ALL_MODELS, 2))
        alpha_bonf = 0.05 / len(pairs)
        print(f"Alpha Bonferroni = 0.05 / {len(pairs)} = {alpha_bonf:.4f}")
        print()
        results = []
        for a, b in pairs:
            stat_mw, p_mw = mannwhitneyu(scores[a], scores[b], alternative="two-sided")
            sig = "*" if p_mw < alpha_bonf else ""
            results.append({"Model A": a, "Model B": b, "p-value": round(p_mw, 6), "Signifikan": "Ya" if sig else "Tidak"})
            print(f"{a} vs {b}: p={p_mw:.6f} {sig}")
        pd.DataFrame(results).to_csv("posthoc_mannwhitney.csv", index=False)
else:
    print("Tidak ada perbedaan signifikan → post-hoc tidak diperlukan.")

POST-HOC: MANN-WHITNEY (dengan Bonferroni correction)
Alpha Bonferroni = 0.05 / 66 = 0.0008

Llama-3.1-8B vs Llama3.2-3B: p=0.304062 
Llama-3.1-8B vs Gemma-2-2B: p=0.254134 
Llama-3.1-8B vs Mistral-7B: p=0.217984 
Llama-3.1-8B vs Qwen2.5-7B: p=0.031793 
Llama-3.1-8B vs Phi-3.5-mini: p=0.010610 
Llama-3.1-8B vs Llama3.2-1B: p=0.000002 *
Llama-3.1-8B vs Qwen2.5-3B: p=0.000000 *
Llama-3.1-8B vs Qwen2.5-1.5B: p=0.000000 *
Llama-3.1-8B vs Qwen2.5-0.5B: p=0.000000 *
Llama-3.1-8B vs SmolLM2-360M: p=0.000000 *
Llama-3.1-8B vs SmolLM2-135M: p=0.000000 *
Llama3.2-3B vs Gemma-2-2B: p=0.955090 
Llama3.2-3B vs Mistral-7B: p=0.854779 
Llama3.2-3B vs Qwen2.5-7B: p=0.307392 
Llama3.2-3B vs Phi-3.5-mini: p=0.150998 
Llama3.2-3B vs Llama3.2-1B: p=0.000001 *
Llama3.2-3B vs Qwen2.5-3B: p=0.000000 *
Llama3.2-3B vs Qwen2.5-1.5B: p=0.000000 *
Llama3.2-3B vs Qwen2.5-0.5B: p=0.000000 *
Llama3.2-3B vs SmolLM2-360M: p=0.000000 *
Llama3.2-3B vs SmolLM2-135M: p=0.000000 *
Gemma-2-2B vs Mistral-7B: p=0.960699 
Gemm

In [9]:
print("=== MODEL TIDAK BERBEDA SIGNIFIKAN DENGAN LLAMA-3.1-8B ===")
referensi = ALL_MODELS[0]
kompetitif = [referensi]

for model in ALL_MODELS[1:]:
    stat_mw, p_mw = mannwhitneyu(scores[referensi], scores[model], alternative="two-sided")
    if p_mw >= alpha_bonf:
        kompetitif.append(model)

print(f"Referensi: {referensi}")
print()
print("Model yang tidak berbeda signifikan dengan model terbaik:")
for i, m in enumerate(kompetitif, 1):
    row = df[df["model"] == m].iloc[0]
    print(f"  #{i} {m} — Final Score: {row['final_score_50_50']:.4f}%")

print()
print("Kesimpulan: Model-model di atas berada pada tier performa")
print("yang setara secara statistik. Pemilihan final mempertimbangkan")
print("efisiensi parameter untuk kebutuhan deployment.")

=== MODEL TIDAK BERBEDA SIGNIFIKAN DENGAN LLAMA-3.1-8B ===
Referensi: Llama-3.1-8B

Model yang tidak berbeda signifikan dengan model terbaik:
  #1 Llama-3.1-8B — Final Score: 49.4784%
  #2 Llama3.2-3B — Final Score: 45.2029%
  #3 Gemma-2-2B — Final Score: 43.8638%
  #4 Mistral-7B — Final Score: 43.6609%
  #5 Qwen2.5-7B — Final Score: 40.9837%
  #6 Phi-3.5-mini — Final Score: 40.5621%

Kesimpulan: Model-model di atas berada pada tier performa
yang setara secara statistik. Pemilihan final mempertimbangkan
efisiensi parameter untuk kebutuhan deployment.


---
## Ringkasan Hasil Uji Statistik

In [10]:
print("=" * 60)
print("RINGKASAN HASIL UJI STATISTIK TUJUAN 1")
print("=" * 60)
print(f"Step 1 — Spearman ρ={rho:.4f}, p={p_spearman:.6f}")
print(f"         → {'Bobot 50/50 terjustifikasi' if p_spearman < 0.05 else 'Bobot 50/50 perlu dikaji ulang'}")
print()
print(f"Step 2 — Uji yang dipakai: {'One-Way ANOVA' if uji == 'anova' else 'Kruskal-Wallis'}")
print(f"         Normalitas   : {'✓ Terpenuhi' if all_normal else '✗ Tidak Terpenuhi'}")
print(f"         Homogenitas  : {'✓ Terpenuhi' if homogen else '✗ Tidak Terpenuhi'}")
print(f"         Hasil        : {'Perbedaan signifikan (p < 0.05)' if signifikan else 'Tidak ada perbedaan signifikan'}")
print(f"         Post-hoc     : {'Tukey HSD' if uji == 'anova' else 'Mann-Whitney + Bonferroni'}")
print()
print(f"TOP 5 model:")
for i, m in enumerate(ALL_MODELS[:5], 1):
    row = df[df["model"] == m].iloc[0]
    print(f"  #{i} {m} — Final Score: {row['final_score_50_50']:.4f}%")

RINGKASAN HASIL UJI STATISTIK TUJUAN 1
Step 1 — Spearman ρ=0.9930, p=0.000000
         → Bobot 50/50 terjustifikasi

Step 2 — Uji yang dipakai: Kruskal-Wallis
         Normalitas   : ✓ Terpenuhi
         Homogenitas  : ✗ Tidak Terpenuhi
         Hasil        : Perbedaan signifikan (p < 0.05)
         Post-hoc     : Mann-Whitney + Bonferroni

TOP 5 model:
  #1 Llama-3.1-8B — Final Score: 49.4784%
  #2 Llama3.2-3B — Final Score: 45.2029%
  #3 Gemma-2-2B — Final Score: 43.8638%
  #4 Mistral-7B — Final Score: 43.6609%
  #5 Qwen2.5-7B — Final Score: 40.9837%
